In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("--------------------------------------------------")
print("🚀 NOYRON 2.0: NEURAL SURROGATE INITIALIZING")
print("--------------------------------------------------")

# ==========================================
# 1. LOAD AND PREPARE THE DATASET
# ==========================================
workspace_dir = r"D:\nyron 2\noyron_workspace"
csv_path = os.path.join(workspace_dir, "labeled_dataset.csv")

print("Loading labeled physics data...")
df = pd.read_csv(csv_path)

# Features (Inputs): Wall Thickness and Number of Holes
X = df[['Wall_Thickness_mm', 'Num_Holes']].values
# Labels (Outputs): Maximum Deflection
y = df[['Max_Deflection_mm']].values

# Neural Networks need normalized data to converge efficiently
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

# Split into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

# Convert arrays to PyTorch Tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train)
X_test_t  = torch.FloatTensor(X_test)
y_test_t  = torch.FloatTensor(y_test)

# ==========================================
# 2. DEFINE THE NEURAL NETWORK
# ==========================================
class NoyronBrain(nn.Module):
    def __init__(self):
        super(NoyronBrain, self).__init__()
        # A lightweight dense network for fast parametric mapping
        self.layer1 = nn.Linear(2, 16)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(16, 16)
        self.relu2 = nn.ReLU()
        self.output = nn.Linear(16, 1) # Single output: Deflection

    def forward(self, x):
        x = self.relu1(self.layer1(x))
        x = self.relu2(self.layer2(x))
        x = self.output(x)
        return x

model = NoyronBrain()
# Mean Squared Error for regression, Adam optimizer for fast learning
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# ==========================================
# 3. TRAINING LOOP
# ==========================================
epochs = 500
print(f"\nInitiating Training Protocol ({epochs} Epochs)...")

for epoch in range(epochs):
    # Forward pass
    predictions = model(X_train_t)
    loss = criterion(predictions, y_train_t)
    
    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

print("✅ Training Complete.")

# ==========================================
# 4. THE ULTIMATE TEST: INSTANT INFERENCE
# ==========================================
model.eval()
print("\n--- AI PHYSICS INFERENCE TEST ---")
print("Testing a design the AI has never seen before...")

# Let's invent a new boom configuration
test_thickness = 3.5  # mm
test_holes = 7

# Normalize the new input using our fitted scaler
new_design = np.array([[test_thickness, test_holes]])
new_design_scaled = scaler_X.transform(new_design)
new_design_tensor = torch.FloatTensor(new_design_scaled)

# Ask the AI to predict the deflection
with torch.no_grad():
    prediction_scaled = model(new_design_tensor)
    
# Un-scale the prediction back to real millimeters
predicted_deflection = scaler_y.inverse_transform(prediction_scaled.numpy())[0][0]

print(f"Design Specs : {test_thickness}mm Wall, {test_holes} Holes")
print(f"AI Prediction: {predicted_deflection:.2f} mm Maximum Deflection")
print("Time taken   : < 0.01 seconds (vs MATLAB's simulation time)")

--------------------------------------------------
🚀 NOYRON 2.0: NEURAL SURROGATE INITIALIZING
--------------------------------------------------
Loading labeled physics data...

Initiating Training Protocol (500 Epochs)...
Epoch [100/500], Loss: 0.0020
Epoch [200/500], Loss: 0.0004
Epoch [300/500], Loss: 0.0001
Epoch [400/500], Loss: 0.0001
Epoch [500/500], Loss: 0.0001
✅ Training Complete.

--- AI PHYSICS INFERENCE TEST ---
Testing a design the AI has never seen before...
Design Specs : 3.5mm Wall, 7 Holes
AI Prediction: 0.00 mm Maximum Deflection
Time taken   : < 0.01 seconds (vs MATLAB's simulation time)


In [3]:
# =========================================================================
# NOYRON 2.0 - PHASE 7: AI-DRIVEN GENERATIVE DESIGN
# The AI searches for the optimal part, then commands the C# engine to build it.
# =========================================================================
import numpy as np

print("--------------------------------------------------")
print("🧠 NOYRON AI: GENERATIVE DESIGN INITIATED")
print("--------------------------------------------------")

# 1. THE ENGINEERING REQUIREMENT
# We need a boom that bends NO MORE than 4.8mm under motor vibration.
MAX_ALLOWED_DEFLECTION = 4.8 

print(f"Mission: Find the lightest possible boom that deflects < {MAX_ALLOWED_DEFLECTION}mm")

best_design = None
# We want to minimize thickness and maximize holes (lower score = lighter part)
best_mass_score = float('inf') 

# 2. THE AI SEARCH LOOP (Exploring 400+ designs in milliseconds)
for thickness in np.arange(2.0, 6.1, 0.1):
    for holes in range(0, 11):
        
        # Ask the AI Surrogate Model to predict the deflection
        X_eval = scaler_X.transform([[thickness, holes]])
        X_tensor = torch.FloatTensor(X_eval)
        
        with torch.no_grad():
            pred_scaled = model(X_tensor)
            pred_defl = scaler_y.inverse_transform(pred_scaled.numpy())[0][0]
        
        # If it survives the physics requirement, evaluate its weight
        if pred_defl <= MAX_ALLOWED_DEFLECTION:
            # Simple heuristic: Thinner walls and more holes = lighter
            mass_score = thickness - (holes * 0.15) 
            
            if mass_score < best_mass_score:
                best_mass_score = mass_score
                best_design = (round(thickness, 1), holes, pred_defl)

opt_thick, opt_holes, opt_defl = best_design

print("\n🏆 OPTIMAL DESIGN FOUND BY AI!")
print(f"Wall Thickness : {opt_thick} mm")
print(f"Hole Count     : {opt_holes}")
print(f"Predicted Bend : {opt_defl:.2f} mm")
print("\nTransmitting coordinates to the Geometry Kernel...")

# =========================================================================
# 3. AUTONOMOUS MANUFACTURE (Calling PicoGK)
# =========================================================================
try:
    boom_length = 150.0
    outer_radius = 20.0
    hole_radius = 8.0

    # Generate Stock
    lat_stock = PicoGK.Lattice()
    lat_stock.AddBeam(
        Vector3(0.0, 0.0, Single(-(boom_length/2))), 
        Vector3(0.0, 0.0, Single((boom_length/2))), 
        Single(outer_radius), Single(outer_radius), Boolean(False)
    )
    vox_part = PicoGK.Voxels(lat_stock)

    # Hollow Core (Using AI's chosen thickness)
    lat_core = PicoGK.Lattice()
    inner_radius = outer_radius - opt_thick
    lat_core.AddBeam(
        Vector3(0.0, 0.0, Single(-(boom_length/2 + 10))), 
        Vector3(0.0, 0.0, Single((boom_length/2 + 10))), 
        Single(inner_radius), Single(inner_radius), Boolean(False)
    )
    vox_part.BoolSubtract(PicoGK.Voxels(lat_core))

    # Drill Holes (Using AI's chosen count)
    if opt_holes > 0:
        lat_holes = PicoGK.Lattice()
        spacing = boom_length / (opt_holes + 1)
        start_z = -(boom_length / 2) + spacing

        for h in range(opt_holes):
            z_pos = start_z + (h * spacing)
            lat_holes.AddBeam(
                Vector3(-40.0, 0.0, Single(z_pos)), 
                Vector3(40.0, 0.0, Single(z_pos)), 
                Single(hole_radius), Single(hole_radius), Boolean(False)
            )
        vox_part.BoolSubtract(PicoGK.Voxels(lat_holes))

    # Export the final AI Masterpiece
    final_stl = os.path.join(workspace_dir, "Noyron_AI_Optimized_Boom.stl")
    PicoGK.Mesh(vox_part).SaveToStlFile(String(final_stl))
    
    print(f"✅ AI Manufacturing Complete! Final STL saved to workspace.")

except Exception as e:
    print(f"❌ Manufacturing Error: {e}")

--------------------------------------------------
🧠 NOYRON AI: GENERATIVE DESIGN INITIATED
--------------------------------------------------
Mission: Find the lightest possible boom that deflects < 4.8mm

🏆 OPTIMAL DESIGN FOUND BY AI!
Wall Thickness : 2.0 mm
Hole Count     : 10
Predicted Bend : 0.01 mm

Transmitting coordinates to the Geometry Kernel...
❌ Manufacturing Error: name 'PicoGK' is not defined
